# Reciter TTS — упаковка модели sherpa-onnx (Kokoro / Orion) для Android

Скачивает готовую модель из релизов **k2-fsa/sherpa-onnx**, добавляет `model.json` и переупаковывает в ZIP (с сохранением дерева `espeak-ng-data/`) для импорта в приложении.

По умолчанию — **Kokoro multi-lang (Orion-like)** (мультиязычный, русский с акцентом, высокое качество 24 кГц). При желании можно переключить на Piper (нативный русский VITS).

Не нужен GPU. Просто скачивание + переупаковка.


## Ячейка 1: Выбор модели


In [ ]:
import os, json, tarfile, zipfile, urllib.request, glob

# --- выбери модель ---
# "kokoro"          : Kokoro v1.0 INT8 (~126 МБ ZIP) — как Orion, но квантованная; русский с акцентом
# "kokoro_fp"       : Kokoro v1.0 FP32 (~310 МБ onnx) — ровно то, что качает AudiFlo/Orion (~400 МБ на диске)
# "kokoro_v11"      : Kokoro v1.1 FP32 — 103 голоса (v1.0 = 53; добавлены в основном китайские)
# "kokoro_v11_int8" : Kokoro v1.1 INT8 — то же, квантованная
# "piper_ru"        : Piper русский (нативный русский без акцента, реальное время)
CHOICE = "kokoro"

REL = "https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models"
KOKORO = {"arch": "sherpa-kokoro", "sr": 24000, "voices": []}
MODELS = {
  "piper_ru": {
     "url": f"{REL}/vits-piper-ru_RU-dmitri-medium.tar.bz2",
     "arch": "sherpa-vits", "sr": 22050,
     "voices": [{"id":"ru_dmitri","locale":"ru-RU","displayName":"Дмитрий","speakerId":0}],
     "display": "Piper Russian (Дмитрий)",
  },
  "kokoro":          {**KOKORO, "url": f"{REL}/kokoro-int8-multi-lang-v1_0.tar.bz2", "display": "Kokoro multi-lang v1.0 int8"},
  "kokoro_fp":       {**KOKORO, "url": f"{REL}/kokoro-multi-lang-v1_0.tar.bz2",      "display": "Kokoro multi-lang v1.0 fp32 (Orion)"},
  "kokoro_v11":      {**KOKORO, "url": f"{REL}/kokoro-multi-lang-v1_1.tar.bz2",      "display": "Kokoro multi-lang v1.1 fp32 (103 голоса)"},
  "kokoro_v11_int8": {**KOKORO, "url": f"{REL}/kokoro-int8-multi-lang-v1_1.tar.bz2", "display": "Kokoro multi-lang v1.1 int8 (103 голоса)"},
}
M = MODELS[CHOICE]
print("выбрано:", M["display"], "->", M["url"])

## Ячейка 2: Скачать + распаковать


In [ ]:
WORK = "/content/sherpa_model"; os.makedirs(WORK, exist_ok=True)
arc = f"{WORK}/model.tar.bz2"
print("качаю..."); urllib.request.urlretrieve(M["url"], arc)
print("распаковка..."); 
with tarfile.open(arc, "r:bz2") as t: t.extractall(WORK)
# найти корневую папку модели (в архиве один верхний каталог)
roots = [d for d in glob.glob(f"{WORK}/*") if os.path.isdir(d)]
ROOT = roots[0]
print("содержимое модели:", sorted(os.listdir(ROOT))[:20])
onnx = [f for f in os.listdir(ROOT) if f.endswith(".onnx")]
print("onnx:", onnx, "| есть espeak-ng-data:", os.path.isdir(f"{ROOT}/espeak-ng-data"),
      "| tokens.txt:", os.path.exists(f"{ROOT}/tokens.txt"))

## Ячейка 2.5 (опционально): прослушать русские голоса до упаковки

Синтезирует русскую фразу несколькими голосами тем же движком sherpa-onnx,
что и приложение (для Kokoro — с `lang="ru"`, ровно как в `SherpaTtsEngine`).
Слушай плееры и выбирай, какие голоса оставить в манифесте.


In [ ]:
# %pip установит CPU-версию sherpa-onnx (~30 c)
%pip install -q sherpa-onnx soundfile

import os, sherpa_onnx, soundfile as sf
from IPython.display import Audio, display

TEXT_RU = "Привет! Это проверка русского синтеза. Съешь ещё этих мягких французских булок, да выпей чаю."

def build_tts(lang=""):
    is_kokoro = CHOICE.startswith("kokoro")
    if is_kokoro:
        lex = ",".join(os.path.join(ROOT, f) for f in os.listdir(ROOT)
                       if f.startswith("lexicon") and f.endswith(".txt"))
        model_cfg = sherpa_onnx.OfflineTtsModelConfig(
            kokoro=sherpa_onnx.OfflineTtsKokoroModelConfig(
                model=os.path.join(ROOT, sorted(f for f in os.listdir(ROOT) if f.endswith(".onnx"))[0]),
                voices=os.path.join(ROOT, "voices.bin"),
                tokens=os.path.join(ROOT, "tokens.txt"),
                data_dir=os.path.join(ROOT, "espeak-ng-data"),
                dict_dir=os.path.join(ROOT, "dict") if os.path.isdir(os.path.join(ROOT, "dict")) else "",
                lexicon=lex,
                lang=lang,   # "ru" => весь текст фонемизируется espeak-ом по-русски
            ),
            num_threads=2,
        )
    else:  # piper / vits
        model_cfg = sherpa_onnx.OfflineTtsModelConfig(
            vits=sherpa_onnx.OfflineTtsVitsModelConfig(
                model=os.path.join(ROOT, sorted(f for f in os.listdir(ROOT) if f.endswith(".onnx"))[0]),
                tokens=os.path.join(ROOT, "tokens.txt"),
                data_dir=os.path.join(ROOT, "espeak-ng-data"),
            ),
            num_threads=2,
        )
    return sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(model=model_cfg))

is_kokoro = CHOICE.startswith("kokoro")
tts = build_tts(lang="ru" if is_kokoro else "")

# Какие голоса послушать: (sid, подпись). Для piper всегда sid=0.
if is_kokoro:
    PROBE = [(2, "af_bella (США, жен.)"), (6, "af_nicole (США, жен.)"),
             (9, "af_sarah (США, жен.)"), (11, "am_adam (США, муж.)"),
             (16, "am_michael (США, муж.)"), (21, "bf_emma (Брит., жен.)"),
             (26, "bm_george (Брит., муж.)")]
else:
    PROBE = [(0, "piper ru")]

os.makedirs("/content/ru_probe", exist_ok=True)
for sid, label in PROBE:
    audio = tts.generate(TEXT_RU, sid=sid, speed=1.0)
    wav = f"/content/ru_probe/sid{sid}.wav"
    sf.write(wav, audio.samples, audio.sample_rate)
    print(f"sid={sid}  {label}  ({len(audio.samples)/audio.sample_rate:.1f} c)")
    display(Audio(wav))

## Ячейка 3: model.json + сборка ZIP


In [ ]:
onnx_name = sorted([f for f in os.listdir(ROOT) if f.endswith(".onnx")])[0]
# для piper в .onnx.json есть sample_rate/num_speakers
meta_sr = M["sr"]
mj = f"{ROOT}/{onnx_name}.json"
if os.path.exists(mj):
    try: meta_sr = json.load(open(mj)).get("audio",{}).get("sample_rate", M["sr"])
    except Exception: pass

KOKORO_V1_0_SPEAKERS = [  # id = индекс; порядок из sherpa-onnx (kokoro-multi-lang-v1_0)
    "af_alloy","af_aoede","af_bella","af_heart","af_jessica","af_kore","af_nicole",
    "af_nova","af_river","af_sarah","af_sky","am_adam","am_echo","am_eric",
    "am_fenrir","am_liam","am_michael","am_onyx","am_puck","am_santa","bf_alice",
    "bf_emma","bf_isabella","bf_lily","bm_daniel","bm_fable","bm_george","bm_lewis",
    "ef_dora","em_alex","ff_siwis","hf_alpha","hf_beta","hm_omega","hm_psi",
    "if_sara","im_nicola","jf_alpha","jf_gongitsune","jf_nezumi","jf_tebukuro",
    "jm_kumo","pf_dora","pm_alex","pm_santa","zf_xiaobei","zf_xiaoni","zf_xiaoxiao",
    "zf_xiaoyi","zm_yunjian","zm_yunxi","zm_yunxia","zm_yunyang",
]
ACCENT = {"a":"США","b":"Брит.","e":"Исп.","f":"Фр.","h":"Хинди","i":"Итал.","j":"Яп.","p":"Порт.","z":"Кит."}

def kokoro_display(name):
    pfx, person = name.split("_", 1)
    acc = ACCENT.get(pfx[0], pfx[0])
    sex = "жен." if pfx[1] == "f" else "муж."
    return f"{person.capitalize()} ({acc}, {sex})"

active_voices = M.get("voices", [])
if CHOICE.startswith("kokoro") and not active_voices:
    vbin = os.path.join(ROOT, "voices.bin")
    if os.path.exists(vbin):
        # sherpa Kokoro voices.bin = float32 [num_voices, 510, 256] -> 522240 B/voice
        num_voices = os.path.getsize(vbin) // (510 * 256 * 4)
        for i in range(num_voices):
            name = KOKORO_V1_0_SPEAKERS[i] if i < len(KOKORO_V1_0_SPEAKERS) else f"voice_{i}"
            disp = kokoro_display(name) if "_" in name else name
            # Каждый голос — дважды: en-US (лексиконный режим) и ru-RU (espeak "ru").
            # Локаль голоса решает, каким языком приложение фонемизирует текст,
            # и даёт чипы "English"/"Русский" в UI + языки в системном TTS.
            active_voices.append({
                "id": name, "locale": "en-US",
                "displayName": disp, "speakerId": i
            })
            active_voices.append({
                "id": f"{name}_ru", "locale": "ru-RU",
                "displayName": f"{disp} · рус. с акцентом", "speakerId": i
            })

model_files = []
for f in os.listdir(ROOT):
    fpath = os.path.join(ROOT, f)
    if os.path.isfile(fpath) and f != "model.json" and not f.startswith("."):
        role = "TALKER" if "onnx" in f else "VOCODER" if "voices.bin" in f else "OTHER"
        model_files.append({
            "filename": f, "role": role, "required": True,
            "sizeMb": round(os.path.getsize(fpath)/1024**2, 2)
        })

manifest = {
  "schemaVersion": 1, "id": f"sherpa-{CHOICE}", "displayName": M["display"],
  "family": "sherpa", "architecture": M["arch"], "sampleRateHz": int(meta_sr),
  "runtime": "sherpa-onnx",
  "tokenizer": {"type": "espeak-ng", "files": ["tokens.txt"]},
  "files": model_files,
  "voices": active_voices,
}
json.dump(manifest, open(f"{ROOT}/model.json","w"), ensure_ascii=False, indent=2)
print("model.json:", json.dumps(manifest, ensure_ascii=False)[:300])

# zip: всё содержимое ROOT под models/ (с сохранением подпапок espeak-ng-data/)
zip_path = f"/content/reciter-{CHOICE}-android.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for dp, _, files in os.walk(ROOT):
        for fn in files:
            full = os.path.join(dp, fn)
            rel = os.path.relpath(full, ROOT)
            zf.write(full, f"models/{rel}")
print("архив:", round(os.path.getsize(zip_path)/1024**2,1), "MB ->", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass